# Gold 04 — Weather and sales

**Grain:** one row per date and store, including dates with zero sales. The daily store Gold table supplies the sales grain. Silver weather is joined by observed date and store. Missing weather stays visible through the has_weather flag. Weather is an observational comparison; a correlation does not establish that weather caused sales.


In [ ]:
import os  # Set Java before Spark starts.
import shutil  # Find Linux Java.
from pathlib import Path  # Build data paths.
from pyspark.sql import SparkSession, functions as F  # Use Spark and column expressions.
from delta import configure_spark_with_delta_pip  # Add Delta Lake to Spark.
java = shutil.which("java")  # Locate WSL Java.
if java is None:  # Require Java.
    raise RuntimeError("Java is unavailable in WSL")  # Explain a startup failure.
os.environ["JAVA_HOME"] = str(Path(java).resolve().parent.parent)  # Avoid the Windows Java path.
project_root = Path("/mnt/f/Big Data/Retail_Lakehouse_Learning")  # Find the project in WSL.
if not project_root.is_dir():  # Check the mounted drive.
    raise FileNotFoundError(project_root)  # Stop on an incorrect path.
builder = (SparkSession.builder.master("local[2]")  # Use two local threads.
    .appName("RetailGoldDailyStoreSales")  # Label this job.
    .config("spark.sql.shuffle.partitions", "4")  # Keep small local shuffles manageable.
    .config("spark.ui.enabled", "false")  # Avoid a second Spark UI port.
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")  # Enable Delta SQL.
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))  # Register Delta.
spark = configure_spark_with_delta_pip(builder).getOrCreate()  # Start or reuse Spark.
print("Spark:", spark.version, "Timezone:", spark.conf.get("spark.sql.session.timeZone"))  # Record session behavior.


In [ ]:
silver_root = project_root / "lakehouse" / "silver" / "data"  # Locate cleaned weather.
gold_root = project_root / "lakehouse" / "gold" / "data"  # Locate checked daily sales.
daily_df = spark.read.format("delta").load(str(gold_root / "daily_store_sales"))  # Read one row per date and store.
weather_df = spark.read.format("delta").load(str(silver_root / "weather"))  # Read typed observations.
daily_count = daily_df.count()  # Record the calendar grain.
if weather_df.count() != weather_df.select("observed_date", "store_id").distinct().count():  # Require unique weather dates/stores.
    raise ValueError("Duplicate Silver weather keys")  # Avoid multiplying revenue.
print("Daily rows:", daily_count, "Weather rows:", weather_df.count())  # Show sources.
weather_df.printSchema()  # Inspect typed weather columns.


In [ ]:
weather_gold_df = (daily_df  # Preserve every daily store row.
    .join(weather_df, (daily_df["date"] == weather_df["observed_date"]) & (daily_df["store_id"] == weather_df["store_id"]), "left")  # Match observation date and store.
    .select(daily_df["date"], daily_df["store_id"], daily_df["store_name"], daily_df["city"], daily_df["region"],  # Keep the sales dimensions.
        weather_df["temperature_c"], weather_df["rainfall_mm"], weather_df["humidity_pct"], weather_df["wind_speed_kmh"], weather_df["weather_code"],  # Keep weather measures.
        weather_df["source"].alias("weather_source"), F.col("observed_date").isNotNull().alias("has_weather"),  # Mark missing weather.
        daily_df["orders"], daily_df["units_sold"], daily_df["revenue"], daily_df["estimated_gross_profit"], daily_df["avg_order_value"]))  # Keep sales KPIs.
if weather_gold_df.count() != daily_count:  # Check the join did not change row count.
    raise ValueError("Weather join changed the daily store grain")  # Stop duplicate or dropped rows.
if weather_gold_df.count() != weather_gold_df.select("date", "store_id").distinct().count():  # Enforce DQ009.
    raise ValueError("Duplicate Gold weather date/store keys")  # Stop before saving.
daily_revenue = daily_df.agg(F.sum("revenue").alias("v")).first().v  # Read the trusted sales total.
weather_revenue = weather_gold_df.agg(F.sum("revenue").alias("v")).first().v  # Check the joined total.
if daily_revenue != weather_revenue:  # A weather join must preserve revenue.
    raise ValueError("Weather sales revenue differs from daily sales")  # Surface multiplicative joins.
print("Weather-sales rows:", weather_gold_df.count(), "Revenue:", weather_revenue)  # Show reconciliation.
print("Missing weather rows:", weather_gold_df.filter(~F.col("has_weather")).count())  # Show observation gaps.
weather_gold_df.orderBy("date", "store_id").show(5, truncate=False)  # Inspect joined values.


In [ ]:
gold_path = gold_root / "weather_sales_analysis"  # Choose Gold Delta location.
gold_path.parent.mkdir(parents=True, exist_ok=True)  # Ensure ignored data parent exists.
weather_gold_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(str(gold_path))  # Commit weather-sales snapshot.
saved_df = spark.read.format("delta").load(str(gold_path))  # Reopen committed Delta data.
if saved_df.count() != daily_count:  # Check saved grain.
    raise ValueError("Saved weather-sales row count differs")  # Surface write issues.
if saved_df.agg(F.sum("revenue").alias("v")).first().v != daily_revenue:  # Check saved revenue.
    raise ValueError("Saved weather-sales revenue differs")  # Surface financial issues.
print("Saved weather_sales_analysis:", saved_df.count(), "Delta log:", (gold_path / "_delta_log").is_dir())  # Confirm Delta commit.
saved_df.printSchema()  # Show typed reporting fields.
